In [25]:
import pandas as pd
from pathlib import Path

# Load experiment 1 datasets
exp_dir = Path('../data/gold/experiment_1')
processes = [p for p in exp_dir.iterdir() if p.is_dir()]

results = []
for p in processes:

    print(p)
    dataset_dir = p / 'datasets'
    if not dataset_dir.exists():
        continue
    
    df_exp_path = dataset_dir / 'df_expanded.parquet'
    if not df_exp_path.exists():
        continue
        
    df = pd.read_parquet(df_exp_path)
    
    # Energy variables (contain 'energy' but NOT 'ef')
    energy_vars = [col for col in df.columns if 'to_model' in col.lower()]

    print(energy_vars)

    num_cols_energy = len(energy_vars) - 1

    
    
    # Number of cases
    case_col = 'case_id' if 'case_id' in df.columns else df.columns[0]
    num_cases = df[case_col].nunique() if case_col in df.columns else None
    
    # Number of activities
    activity_col = 'activity_log'
    num_activities = df[activity_col].nunique() if activity_col in df.columns else None

    # Max and min date, complete duration
    time_cols = [col for col in df.columns if 'time' in col.lower() or 'date' in col.lower()]
    time_col = time_cols[0] if time_cols else None
    if 'timestamp' in df.columns:
        time_col = 'timestamp'
    elif 'time:timestamp' in df.columns:
        time_col = 'time:timestamp'
        
    if time_col:
        # Try converting to datetime if not already
        df[time_col] = pd.to_datetime(df[time_col], errors='coerce')
        min_date = df[time_col].min()
        max_date = df[time_col].max()
        complete_date = max_date - min_date
        # Convert timedelta to total hours (rounded to 2 decimal places for readability)
        hours = round(complete_date.total_seconds() / 3600, 2) if pd.notnull(complete_date) else None
    else:
        min_date, max_date, hours = None, None, None
        
    # Number of rows in expanded
    num_rows = len(df)
    
    results.append({
        'dataset': p.name,
        'number of cases': num_cases,
        'number of activities': num_activities,
        'number of time series': num_cols_energy,
        'hours': int(round(hours)) if hours is not None else None,
        #'observations': num_rows
    })

# Sort and display the DataFrame
df_info = pd.DataFrame(results).sort_values(by='dataset')
display(df_info)

# Output a single LaTeX table for all datasets
print("\\begin{table}[h]")
print("\\centering")
print("\\caption{Dataset Information Overview}")
print(df_info.to_latex(index=False).replace('_', '\\_'))
print("\\end{table}")


../data/gold/experiment_1/process_4
['temp_Auslauf_EG_(WT2)_5s_energy_to_model', 'temp_Einlauf_EG_(WT_2)_5s_energy_to_model', 'flow_Kuehlturmwasser_30120FT701_5s_energy_to_model', 'flow_Kaltwasser_(WT7)_5s_energy_to_model', 'Vorlaufpumpe_30110FT301_5s_energy_to_model', 'vor_Vorwärmer_(WT_2)_5s_energy_to_model', 'Kuehlturmwassertemp_(WT6)_5s_energy_to_model', 'Kaltwassertemp_(WT_7)_5s_energy_to_model', 'nach_Kuehler_(WT7)_5s_energy_to_model', 'temp_nach_Kuehlturmkuehler_(WT6)_5s_energy_to_model', 'Fuellstand_Steriltank_30140LT001_5s_energy_to_model', 'Fuellstand_Steriltank_30141LT001_5s_energy_to_model', 'flow_Dampf_WT3a/5a)_5s_energy_to_model', 'flow_Heisswasser_30120FT721(WT5a)_5s_energy_to_model', 'temp_nach_WR2,_vor_Druckerhoehungspumpe_(WT4)_5s_energy_to_model', 'temp_nach_Erhitzer_(WT5)_5s_energy_to_model', 'temp_nach_WR2_(WT2)_5s_energy_to_model', 'temp_nach_Austauscher_2_(WT4)_5s_energy_to_model', 'Druck_HW_Anwaermer_(WT3a)_5s_energy_to_model', 'temp_HW_Anwaermer_(WT3a)_5s_energ

,dataset,number of cases,number of activities,number of time series,hours
1,process_1,155199,20,5,8757
3,process_2,368493,12,4,512
4,process_3,368493,12,4,512
0,process_4,53,19,27,705
2,process_5,53,19,27,705


\begin{table}[h]
\centering
\caption{Dataset Information Overview}
\begin{tabular}{lrrrr}
\toprule
dataset & number of cases & number of activities & number of time series & hours \\
\midrule
process\_1 & 155199 & 20 & 5 & 8757 \\
process\_2 & 368493 & 12 & 4 & 512 \\
process\_3 & 368493 & 12 & 4 & 512 \\
process\_4 & 53 & 19 & 27 & 705 \\
process\_5 & 53 & 19 & 27 & 705 \\
\bottomrule
\end{tabular}

\end{table}


In [26]:
# ── Config ────────────────────────────────────────────────────────────────────
# Set the experiment number to load (picks the latest run automatically)
EXPERIMENT = 505
# Which evaluation split to display: 'train' or 'test'
# Both live in the same row — train_* vs test_* column prefixes.
SPLIT = 'test'

In [27]:
import pandas as pd
from pathlib import Path

results_root = Path('..') / 'results'

# Find the latest run folder for the given experiment
prefix = f'experiment_{EXPERIMENT}_'
runs = sorted([d for d in results_root.iterdir() if d.is_dir() and d.name.startswith(prefix)])
assert runs, f'No runs found for experiment {EXPERIMENT}'
run_dir = runs[-1]
print(f'Loading: {run_dir.name}')

df = pd.read_parquet(run_dir / 'process_eval_results.parquet')
# The 'split' column marks the training set used (always 'TRAIN' or 'ALL DATA').
# Train vs test evaluation results are distinguished by column prefix: train_* vs test_*.
# Check the requested prefix exists.
split_prefix = SPLIT.lower() + '_'
available_prefixes = set()
for c in df.columns:
    for p in ('train_', 'test_'):
        if c.startswith(p):
            available_prefixes.add(p.rstrip('_'))
print(f'Available column prefixes: {sorted(available_prefixes)}')
assert split_prefix.rstrip('_') in available_prefixes, \
    f'No columns with prefix "{split_prefix}" found. Available: {sorted(available_prefixes)}'

print(f'Split prefix: {split_prefix} | {len(df)} rows | processes: {sorted(df["process"].unique())}')

Loading: experiment_505_20260629_160359
Available column prefixes: ['test', 'train']
Split prefix: test_ | 50 rows | processes: ['process_1', 'process_2', 'process_3', 'process_4', 'process_5']


In [28]:
# ── Mode display labels (mirrors modelling.py _display_mode) ─────────────────
def display_mode(m):
    m = str(m)
    if not m.startswith('petri_net_'):
        return m
    rest = m[len('petri_net_'):]
    if rest.endswith('_ml_plus_global'):
        return rest[:-len('_ml_plus_global')] + ' / ml_global'
    if rest.endswith('_ml_plus_per_act'):
        return rest[:-len('_ml_plus_per_act')] + ' / ml_local'
    return rest + ' / baseline'

df['mode_label'] = df['mode'].map(display_mode)

In [29]:
# ── Metrics for the results table ────────────────────────────────────────────
# activity_duration_mae:  MAE of per-activity mean durations in minutes
# activity_duration_wape: weighted abs % error of per-activity mean durations (0 = perfect)
# activity_duration_rmse: RMSE of per-activity mean durations in minutes
# edge_f1_score:          control-flow edge F1 on directly-follows graph (1 = perfect)
# fitness / precision:    conformance values via token replay (1 = perfect)

split_prefix = SPLIT.lower() + '_'

# Duration metrics: prefer MAE/WAPE/RMSE (new pipeline), fall back to MAPE for old parquets
_has_wape = (split_prefix + 'duration_metrics_activity_duration_wape') in df.columns

if _has_wape:
    _dur_cols = [
        split_prefix + 'duration_metrics_activity_duration_mae',
        split_prefix + 'duration_metrics_activity_duration_wape',
        split_prefix + 'duration_metrics_activity_duration_rmse',
    ]
    _dur_labels = {
        split_prefix + 'duration_metrics_activity_duration_mae':  r'\makecell{Duration\\MAE (min)}',
        split_prefix + 'duration_metrics_activity_duration_wape': r'\makecell{Duration\\WAPE}',
        split_prefix + 'duration_metrics_activity_duration_rmse': r'\makecell{Duration\\RMSE (min)}',
    }
else:
    _dur_cols = [split_prefix + 'duration_metrics_activity_duration_error']
    _dur_labels = {split_prefix + 'duration_metrics_activity_duration_error': r'\makecell{Duration\\MAPE}'}

METRIC_BASES_ORDERED = (
    _dur_cols
    + [
        split_prefix + 'control_flow_metrics_edge_f1_score',
        split_prefix + 'conformance_metrics_fitness',
        split_prefix + 'conformance_metrics_precision',
    ]
)
metric_cols = [c for c in METRIC_BASES_ORDERED if c in df.columns]

METRIC_LABELS = {
    **_dur_labels,
    split_prefix + 'control_flow_metrics_edge_f1_score': r'\makecell{Edge\\F1 Score}',
    split_prefix + 'conformance_metrics_fitness':        r'\makecell{Fitness}',
    split_prefix + 'conformance_metrics_precision':      r'\makecell{Precision}',
}

processes = sorted(df['process'].unique())
print(f'Duration cols: {_dur_cols}')
print(f'All metrics: {metric_cols}')
print(f'Processes: {processes}')
print(f'Modes: {df["mode_label"].unique().tolist()}')

Duration cols: ['test_duration_metrics_activity_duration_mae', 'test_duration_metrics_activity_duration_wape', 'test_duration_metrics_activity_duration_rmse']
All metrics: ['test_duration_metrics_activity_duration_mae', 'test_duration_metrics_activity_duration_wape', 'test_duration_metrics_activity_duration_rmse', 'test_control_flow_metrics_edge_f1_score', 'test_conformance_metrics_fitness', 'test_conformance_metrics_precision']
Processes: ['process_1', 'process_2', 'process_3', 'process_4', 'process_5']
Modes: ['statistical', 'alpha / baseline', 'heuristic / baseline', 'inductive / baseline', 'alpha / ml_global', 'alpha / ml_local', 'heuristic / ml_global', 'heuristic / ml_local', 'inductive / ml_global', 'inductive / ml_local']


In [30]:
# ── Build table: rows=(process, process model, duration pred), cols=metrics ───

def pick_best_algo(proc_df, split_prefix):
    """Best petri net algo among {heuristic, inductive, ilp} by combined metric score."""
    candidates = ['heuristic', 'inductive', 'ilp']
    base_modes = [f'petri_net_{a}' for a in candidates]
    sub = proc_df[proc_df['mode'].isin(base_modes)]
    if sub.empty:
        return candidates[0]
    _score_cols = [c for c in [
        split_prefix + 'conformance_metrics_fitness',
        split_prefix + 'conformance_metrics_precision',
        split_prefix + 'control_flow_metrics_edge_f1_score',
    ] if c in sub.columns]
    scores = pd.Series(0.0, index=sub.index)
    for c in _score_cols:
        scores = scores + sub[c].fillna(0)
    best_mode = sub.loc[scores.idxmax(), 'mode']
    return best_mode.replace('petri_net_', '')

def make_mode_label(mode, best_algo):
    pn = f'{best_algo} petri net'
    if mode == 'petri_net_alpha':
        return 'alpha petri net / baseline'
    if mode == f'petri_net_{best_algo}':
        return f'{pn} / baseline'
    if mode == f'petri_net_{best_algo}_ml_plus_global':
        return f'{pn} / ml_global'
    if mode == f'petri_net_{best_algo}_ml_plus_per_act':
        return f'{pn} / ml_local'
    return mode

def split_mode(label):
    if ' / ' in label:
        left, right = label.split(' / ', 1)
        return left.strip(), right.strip()
    return label, '—'

frames = []
for proc in processes:
    sub = df[df['process'] == proc].copy()
    best_algo = pick_best_algo(sub, split_prefix)
    print(f'{proc}: best_algo = {best_algo}')

    modes_to_show = [
        'petri_net_alpha',
        f'petri_net_{best_algo}',
        f'petri_net_{best_algo}_ml_plus_global',
        f'petri_net_{best_algo}_ml_plus_per_act',
    ]
    sub = sub[sub['mode'].isin(modes_to_show)].copy()
    sub['mode_label'] = sub['mode'].apply(lambda m: make_mode_label(m, best_algo))

    # ml_local first, ml_global second, baseline (best net) third, alpha last
    pn = f'{best_algo} petri net'
    row_order = [
        f'{pn} / ml_local',
        f'{pn} / ml_global',
        f'{pn} / baseline',
        'alpha petri net / baseline',
    ]
    sub = sub.set_index('mode_label')[metric_cols]
    sub = sub.reindex([r for r in row_order if r in sub.index])
    sub.columns = [METRIC_LABELS.get(c, c) for c in sub.columns]
    pmodel, dpred = zip(*[split_mode(m) for m in sub.index])
    sub.index = pd.MultiIndex.from_arrays(
        [[proc] * len(sub), list(pmodel), list(dpred)],
        names=['Process', 'Process Model', 'Duration Pred.']
    )
    frames.append(sub)

table = pd.concat(frames)
table

process_1: best_algo = heuristic
process_2: best_algo = heuristic
process_3: best_algo = heuristic
process_4: best_algo = heuristic
process_5: best_algo = heuristic


\makecell{Duration\\MAE (min)}  \
Process   Process Model       Duration Pred.                                   
process_1 heuristic petri net ml_local                              0.117859   
                              ml_global                             0.200576   
                              baseline                              0.985203   
          alpha petri net     baseline                              0.985203   
process_2 heuristic petri net ml_local                              4.507381   
                              ml_global                             3.013872   
                              baseline                              7.683950   
          alpha petri net     baseline                             27.125716   
process_3 heuristic petri net ml_local                              4.280978   
                              ml_global                             4.173959   
                              baseline                              3.772251   
          alpha petri net     baseline                             31.002561   
process_4 heuristic petri net ml_local                              4.091054   
                              ml_global                             3.595585   
                              baseline                              3.717815   
          alpha petri net     baseline                              8.199242   
process_5 heuristic petri net ml_local                              4.091054   
                              ml_global                             3.595585   
                              baseline                              3.717815   
          alpha petri net     baseline                              8.199242   

                                              \makecell{Duration\\WAPE}  \
Process   Process Model       Duration Pred.                              
process_1 heuristic petri net ml_local                         3.579618   
                              ml_global                        9.254257   
                              baseline                        33.081765   
          alpha petri net     baseline                        33.081765   
process_2 heuristic petri net ml_local                        24.700011   
                              ml_global                       22.189246   
                              baseline                        55.589791   
          alpha petri net     baseline                       103.685879   
process_3 heuristic petri net ml_local                        31.384028   
                              ml_global                       27.114529   
                              baseline                        33.856901   
          alpha petri net     baseline                        54.263526   
process_4 heuristic petri net ml_local                        39.705438   
                              ml_global                       34.896697   
                              baseline                        36.082991   
          alpha petri net     baseline                        44.121985   
process_5 heuristic petri net ml_local                        39.705438   
                              ml_global                       34.896697   
                              baseline                        36.082991   
          alpha petri net     baseline                        44.121985   

                                              \makecell{Duration\\RMSE (min)}  \
Process   Process Model       Duration Pred.                                    
process_1 heuristic petri net ml_local                               0.193549   
                              ml_global                              0.273868   
                              baseline                               1.602989   
          alpha petri net     baseline                               1.602989   
process_2 heuristic petri net ml_local                               8.831194   
                              ml_global                            

In [31]:
# ── LaTeX output ──────────────────────────────────────────────────────────────
# Requires \usepackage{makecell}, \usepackage{booktabs}, \usepackage{multirow} in preamble.
import re

def _is_higher_better(col_label):
    return any(kw in col_label for kw in ['F1 Score', 'Fitness', 'Precision'])

# ── Bold best value per metric per process ────────────────────────────────────
str_table = pd.DataFrame(index=table.index, columns=table.columns, dtype=object)
for proc in processes:
    proc_mask = table.index.get_level_values('Process') == proc
    proc_rows = table[proc_mask]
    for col in table.columns:
        vals = proc_rows[col].dropna()
        if vals.empty:
            for idx in proc_rows.index:
                str_table.loc[idx, col] = ''
            continue
        best_val = vals.max() if _is_higher_better(col) else vals.min()
        for idx in proc_rows.index:
            val = table.loc[idx, col]
            if pd.isna(val):
                str_table.loc[idx, col] = ''
                continue
            fmt = f'{val:.3f}'
            str_table.loc[idx, col] = (r'\textbf{' + fmt + r'}') if abs(val - best_val) < 1e-6 else fmt

# ── Column format: vertical lines between metrics ─────────────────────────────
n_metrics = len(table.columns)
col_format = 'lll|' + '|'.join(['c'] * n_metrics)

# ── Generate LaTeX ─────────────────────────────────────────────────────────────
latex = str_table.to_latex(
    multicolumn=True,
    multicolumn_format='c',
    multirow=True,
    escape=False,
    column_format=col_format,
    caption=(
        f'Results for experiment {EXPERIMENT} ({SPLIT} evaluation). '
        r'Rows: alpha = alpha miner (baseline); best petri net = best of heuristic/inductive per process. '
        r'Activity JS Div.\ and Duration WAPE/MAE/RMSE: lower is better. '
        r'Edge F1 Score, Fitness, Precision: higher is better (1 = best). '
        r'\textbf{Bold} = best value per process per metric.'
    ),
    label=f'tab:exp{EXPERIMENT}_{SPLIT}',
    position='H',
)

# Remove \cline lines (generated by pandas for MultiIndex, replaced by \midrule below)
latex = re.sub(r'\s*\\cline\{[^}]+\}', '', latex)

# Insert \midrule between process groups (detect first row of each group via \multirow + process_)
latex_lines = latex.split('\n')
new_lines = []
proc_count = 0
for line in latex_lines:
    if re.search(r'\\multirow.*\{process_', line):
        if proc_count > 0:
            new_lines.append(r'\midrule')
        proc_count += 1
    new_lines.append(line)
latex = '\n'.join(new_lines)

print(latex.replace('_', r'\_'))

\begin{table}[H]
\caption{Results for experiment 505 (test evaluation). Rows: alpha = alpha miner (baseline); best petri net = best of heuristic/inductive per process. Activity JS Div.\ and Duration WAPE/MAE/RMSE: lower is better. Edge F1 Score, Fitness, Precision: higher is better (1 = best). \textbf{Bold} = best value per process per metric.}
\label{tab:exp505\_test}
\begin{tabular}{lll|c|c|c|c|c|c}
\toprule
 &  &  & \makecell{Duration\\MAE (min)} & \makecell{Duration\\WAPE} & \makecell{Duration\\RMSE (min)} & \makecell{Edge\\F1 Score} & \makecell{Fitness} & \makecell{Precision} \\
Process & Process Model & Duration Pred. &  &  &  &  &  &  \\
\midrule
\multirow[t]{4}{*}{process\_1} & \multirow[t]{3}{*}{heuristic petri net} & ml\_local & \textbf{0.118} & \textbf{3.580} & \textbf{0.194} & \textbf{0.615} & \textbf{1.000} & \textbf{1.000} \\
 &  & ml\_global & 0.201 & 9.254 & 0.274 & \textbf{0.615} & \textbf{1.000} & \textbf{1.000} \\
 &  & baseline & 0.985 & 33.082 & 1.603 & \textbf{0.6

# Evaluation energy

In [32]:
df_energy_results = pd.read_parquet(run_dir / 'summary_by_approach.parquet')

df_energy_results = df_energy_results[['Approach', 'MAE_TEST', 'RMSE_TEST', 'WAPE_TEST']]

df_energy_results

,Approach,MAE_TEST,RMSE_TEST,WAPE_TEST
0,Baseline,2.4412,2.6306,19.3307
1,DTW + Ext. Factors + Prev Act,1.2426,1.3922,2.8927
2,DTW + Ext. Factors + Prev Act (autoreg),2.9459,3.1618,7.1229
3,DTW + Seq2Seq,1.4187,1.6018,9.4349
4,DTW + Seq2Seq + Ext. Factors + Prev Act,4.4874,4.6323,9.3835
5,DTW + Seq2Seq + Ext. Factors + Prev Act (autoreg),4.4904,4.5909,9.1268
6,DTW + pos,1.3094,1.4069,8.9680


In [33]:
import pandas as pd

# Load data
df_energy_results = pd.read_parquet(run_dir / "summary_train_test.parquet")

# Group by Process and Approach, then take median of the selected metrics
df_grouped = (
    df_energy_results
    .groupby(["Process", "Approach"])[["MAE_TEST", "RMSE_TEST", "WAPE_TEST"]]
    .median()
    .reset_index()
)

df_grouped

,Process,Approach,MAE_TEST,RMSE_TEST,WAPE_TEST
0,process_1,Baseline,0.00000,0.00000,85.39000
1,process_1,DTW + Seq2Seq,0.00000,0.00000,7.64510
2,process_1,DTW + pos,0.00000,0.00000,8.48345
3,process_2,Baseline,97.74170,115.45780,135.58460
4,process_2,DTW + Ext. Factors + Prev Act,79.14870,93.18890,85.14250
5,process_2,DTW + Ext. Factors + Prev Act (autoreg),92.99480,105.55390,80.94950
6,process_2,DTW + Seq2Seq,65.01180,75.95370,69.42840
7,process_2,DTW + Seq2Seq + Ext. Factors + Prev Act,89.65260,101.05920,113.54720
8,process_2,DTW + Seq2Seq + Ext. Factors + Prev Act (autoreg),89.46460,100.91820,115.71470
9,process_2,DTW + pos,89.66160,100.73930,48.16440


In [34]:
import pandas as pd
import re

# Load data
df_energy_results = pd.read_parquet(run_dir / "summary_train_test.parquet")

# Group by Process and Approach, then take median
table = (
    df_energy_results
    .groupby(["Process", "Approach"])[["MAE_TEST", "RMSE_TEST", "WAPE_TEST"]]
    .median()
    .sort_index()
)

processes = table.index.get_level_values("Process").unique()

# Lower is better for all these metrics
def _is_higher_better(col_label):
    return False

# ── Bold best value per metric per process ────────────────────────────────────
str_table = pd.DataFrame(index=table.index, columns=table.columns, dtype=object)

for proc in processes:
    proc_mask = table.index.get_level_values("Process") == proc
    proc_rows = table[proc_mask]
    
    for col in table.columns:
        vals = proc_rows[col].dropna()
        if vals.empty:
            for idx in proc_rows.index:
                str_table.loc[idx, col] = ""
            continue
        
        best_val = vals.max() if _is_higher_better(col) else vals.min()
        
        for idx in proc_rows.index:
            val = table.loc[idx, col]
            if pd.isna(val):
                str_table.loc[idx, col] = ""
                continue
            
            fmt = f"{val:.3f}"
            str_table.loc[idx, col] = (
                r"\textbf{" + fmt + r"}"
                if abs(val - best_val) < 1e-6
                else fmt
            )

# ── Column format ─────────────────────────────────────────────────────────────
n_metrics = len(table.columns)
col_format = "ll|" + "|".join(["c"] * n_metrics)

# ── Generate LaTeX ────────────────────────────────────────────────────────────
latex = str_table.to_latex(
    multicolumn=True,
    multicolumn_format="c",
    multirow=True,
    escape=False,
    column_format=col_format,
    caption=(
        r"Median test results grouped by Process and Approach. "
        r"Metrics shown: MAE\_TEST, RMSE\_TEST, and WAPE\_TEST. "
        r"Lower values are better. "
        r"\textbf{Bold} indicates the best value per process and metric."
    ),
    label="tab:energy_results_median",
    position="H",
)

# Remove \cline lines
latex = re.sub(r"\s*\\cline\{[^}]+\}", "", latex)

# Insert \midrule between process groups
latex_lines = latex.split("\n")
new_lines = []
proc_count = 0

for line in latex_lines:
    if re.search(r"\\multirow", line):
        if proc_count > 0:
            new_lines.append(r"\midrule")
        proc_count += 1
    new_lines.append(line)

latex = "\n".join(new_lines)

print(latex.replace("_", r"\_"))

\begin{table}[H]
\caption{Median test results grouped by Process and Approach. Metrics shown: MAE\\_TEST, RMSE\\_TEST, and WAPE\\_TEST. Lower values are better. \textbf{Bold} indicates the best value per process and metric.}
\label{tab:energy\_results\_median}
\begin{tabular}{ll|c|c|c}
\toprule
 &  & MAE\_TEST & RMSE\_TEST & WAPE\_TEST \\
Process & Approach &  &  &  \\
\midrule
\multirow[t]{3}{*}{process\_1} & Baseline & \textbf{0.000} & \textbf{0.000} & 85.390 \\
 & DTW + Seq2Seq & \textbf{0.000} & \textbf{0.000} & \textbf{7.645} \\
 & DTW + pos & \textbf{0.000} & \textbf{0.000} & 8.483 \\
\midrule
\multirow[t]{7}{*}{process\_2} & Baseline & 97.742 & 115.458 & 135.585 \\
 & DTW + Ext. Factors + Prev Act & 79.149 & 93.189 & 85.142 \\
 & DTW + Ext. Factors + Prev Act (autoreg) & 92.995 & 105.554 & 80.950 \\
 & DTW + Seq2Seq & \textbf{65.012} & \textbf{75.954} & 69.428 \\
 & DTW + Seq2Seq + Ext. Factors + Prev Act & 89.653 & 101.059 & 113.547 \\
 & DTW + Seq2Seq + Ext. Factors + Prev Act

In [35]:
df_energy_results = pd.read_parquet(run_dir / 'summary_train_test.parquet')

#df_energy_results = df_energy_results[['Approach', 'MAE_TEST', 'RMSE_TEST', 'WAPE_TEST']]

df_energy_results

,Process,Sensor,Approach,MAE_TRAIN,RMSE_TRAIN,WAPE_TRAIN,R2_TRAIN,MAE_TEST,RMSE_TEST,WAPE_TEST,R2_TEST
0,process_1,autoclave_cooling_water_demand_kW_energy_to_model,Baseline,33.0395,33.3506,70.4301,-0.0199,26.6229,28.0244,70.7800,-0.0793
1,process_1,autoclave_cooling_water_demand_kW_energy_to_model,DTW + Seq2Seq,5.1205,5.3210,11.0264,0.9723,2.6059,2.9354,6.9713,0.9881
2,process_1,autoclave_cooling_water_demand_kW_energy_to_model,DTW + pos,5.3657,5.4802,11.7240,0.9720,2.6675,2.9440,7.2081,0.9878
3,process_1,autoclave_steam_demand_kW_energy_to_model,Baseline,0.0000,0.0000,NaN,1.0000,0.0000,0.0000,NaN,1.0000
4,process_1,autoclave_steam_demand_kW_energy_to_model,DTW + Seq2Seq,0.0000,0.0000,NaN,0.0000,0.0000,0.0000,NaN,0.0000
...,...,...,...,...,...,...,...,...,...,...,...
475,process_5,vor_Vorwärmer_(WT_2)_5s_energy_to_model,DTW + Ext. Factors + Prev Act (autoreg),0.5565,0.6151,0.9523,-4.4502,1.1656,1.2204,1.9906,-6.8603
476,process_5,vor_Vorwärmer_(WT_2)_5s_energy_to_model,DTW + Seq2Seq,0.5790,0.5976,0.9685,-4.2214,0.7132,0.8914,1.1808,-4.3655
477,process_5,vor_Vorwärmer_(WT_2)_5s_energy_to_model,DTW + Seq2Seq + Ext. Factors + Prev Act,0.4543,0.4679,0.7573,-4.1724,0.9604,1.1323,1.6273,-6.4454
478,process_5,vor_Vorwärmer_(WT_2)_5s_energy_to_model,DTW + Seq2Seq + Ext. Factors + Prev Act (autoreg),0.5082,0.5473,0.8469,-6.7608,1.0081,1.1415,1.7078,-6.1715


In [36]:
df_energy_results['Approach'].unique()

array(['Baseline', 'DTW + Seq2Seq', 'DTW + pos',
       'DTW + Ext. Factors + Prev Act',
       'DTW + Ext. Factors + Prev Act (autoreg)',
       'DTW + Seq2Seq + Ext. Factors + Prev Act',
       'DTW + Seq2Seq + Ext. Factors + Prev Act (autoreg)'], dtype=object)

In [37]:
df_energy_results['Process'].value_counts()

Process
process_5    196
process_4    196
process_3     35
process_2     35
process_1     18
Name: count, dtype: int64

# Joint Duration + Profile Evaluation

Three tables:
1. **All together** — median sMAE / sRMSE / WAPE per curve approach, aggregated over all processes (best SimMode per process used for durations).
2. **Per simulation mode** — rows = simulation mode sorted by duration WAPE, cols = curve metrics; one table per process × curve approach (requires `curve_joint_duration_eval_per_mode.parquet` from a fresh pipeline run).
3. **Combined** — the process-results table from above extended with curve columns for Baseline and best curve approach; shows that curve quality degrades together with duration quality.

In [38]:
import re as _re

# ── Joint eval ALL TOGETHER — median by Approach across all processes ─────────
_jall_path = run_dir / 'curve_joint_duration_eval_results.parquet'
if not _jall_path.exists():
    print(f'Not found: {_jall_path}  (run the pipeline first)')
else:
    _jall = pd.read_parquet(_jall_path)
    _jm = [m for m in ['sMAE', 'sRMSE', 'WAPE'] if m in _jall.columns]

    # Median per Approach (across all processes, sensors, activities)
    _jall_agg = _jall.groupby('Approach')[_jm].median().round(4)
    _sort_col  = 'WAPE' if 'WAPE' in _jall_agg.columns else _jm[0]
    _jall_agg  = _jall_agg.sort_values(_sort_col)
    display(_jall_agg)

    # Bold the best (lowest) per column
    _jall_str = pd.DataFrame(index=_jall_agg.index, columns=_jall_agg.columns, dtype=object)
    for _c in _jall_agg.columns:
        _best = _jall_agg[_c].dropna().min()
        for _idx in _jall_agg.index:
            _v = _jall_agg.loc[_idx, _c]
            if pd.isna(_v):
                _jall_str.loc[_idx, _c] = ''
            else:
                _fmt = f'{_v:.3f}'
                _jall_str.loc[_idx, _c] = (r'\textbf{' + _fmt + r'}') if abs(_v - _best) < 1e-6 else _fmt

    _jall_str.columns = [
        r'\makecell{sMAE}' if c == 'sMAE' else
        r'\makecell{sRMSE}' if c == 'sRMSE' else
        r'\makecell{WAPE (\%)}' for c in _jall_str.columns
    ]
    _jall_latex = _jall_str.to_latex(
        escape=False,
        column_format='l|' + '|'.join(['c'] * len(_jm)),
        caption=(
            r'Joint duration + profile evaluation: median sMAE / sRMSE / WAPE per curve approach, '
            r'aggregated over all processes and sensors. '
            r'Simulated durations come from the best process model per process (lowest test duration WAPE). '
            r'Lower is better. \textbf{Bold} = best approach.'
        ),
        label=f'tab:joint_eval_all_{EXPERIMENT}',
        position='H',
    )
    print(_jall_latex.replace('_', r'\_'))

,sMAE,sRMSE,WAPE
Approach,,,
DTW + Ext. Factors + Prev Act,1.7336,1.9396,2.9217
DTW + pos,2.7134,2.9638,8.8569
DTW + Seq2Seq,3.8811,4.1402,9.2802
DTW + Seq2Seq + Ext. Factors + Prev Act,4.4845,4.7347,10.5154
Baseline,3.6432,3.7780,15.0706


\begin{table}[H]
\caption{Joint duration + profile evaluation: median sMAE / sRMSE / WAPE per curve approach, aggregated over all processes and sensors. Simulated durations come from the best process model per process (lowest test duration WAPE). Lower is better. \textbf{Bold} = best approach.}
\label{tab:joint\_eval\_all\_505}
\begin{tabular}{l|c|c|c}
\toprule
 & \makecell{sMAE} & \makecell{sRMSE} & \makecell{WAPE (\%)} \\
Approach &  &  &  \\
\midrule
DTW + Ext. Factors + Prev Act & \textbf{1.734} & \textbf{1.940} & \textbf{2.922} \\
DTW + pos & 2.713 & 2.964 & 8.857 \\
DTW + Seq2Seq & 3.881 & 4.140 & 9.280 \\
DTW + Seq2Seq + Ext. Factors + Prev Act & 4.484 & 4.735 & 10.515 \\
Baseline & 3.643 & 3.778 & 15.071 \\
\bottomrule
\end{tabular}
\end{table}



In [39]:
# ── Joint eval PER SIMULATION MODE ───────────────────────────────────────────
# Rows = simulation mode sorted by duration WAPE (best first).
# Columns = sMAE / sRMSE / WAPE for Baseline and best curve approach per process.
# Requires curve_joint_duration_eval_per_mode.parquet (generated by the new pipeline block).

_jpm_path = run_dir / 'curve_joint_duration_eval_per_mode.parquet'
if not _jpm_path.exists():
    print(f'Per-mode joint eval not found at:\n  {_jpm_path}\n'
          'Re-run the pipeline after the modelling.py update to generate this file.')
else:
    _jpm = pd.read_parquet(_jpm_path)
    _jpm_metrics = [m for m in ['sMAE', 'sRMSE', 'WAPE'] if m in _jpm.columns]

    # Duration WAPE per (process, mode) for row ordering
    _pe2 = pd.read_parquet(run_dir / 'process_eval_results.parquet')
    _wc2 = next((c for c in ['test_duration_metrics_activity_duration_wape',
                              'test_duration_metrics_activity_duration_mae']
                 if c in _pe2.columns), None)
    _mwape = {}
    if _wc2:
        for (_pp, _pm), _gg in _pe2.groupby(['process', 'mode']):
            _vv = _gg[_wc2].dropna()
            if not _vv.empty:
                _mwape[(_pp, _pm)] = float(_vv.min())

    # Best curve approach per process (lowest median WAPE_TEST)
    _stp = run_dir / 'summary_train_test.parquet'
    _best_appr_pm = {}
    if _stp.exists():
        _stdf = pd.read_parquet(_stp)
        for _pr, _sg in _stdf.groupby('Process'):
            _pa = _sg.groupby('Approach')['WAPE_TEST'].median()
            if not _pa.empty:
                _best_appr_pm[_pr] = _pa.idxmin()

    def _fmt_sim_mode(m):
        m = str(m)
        if not m.startswith('petri_net_'):
            return m.replace('_', r'\_')
        rest = m[len('petri_net_'):]
        if rest.endswith('_ml_plus_per_act'):
            return rest[:-len('_ml_plus_per_act')].replace('_', r'\_') + r' / ml\_local'
        if rest.endswith('_ml_plus_global'):
            return rest[:-len('_ml_plus_global')].replace('_', r'\_') + r' / ml\_global'
        return rest.replace('_', r'\_') + r' / baseline'

    _pm_latex_parts = []

    for _proc in processes:
        _psub = _jpm[_jpm['Process'] == _proc]
        if _psub.empty:
            continue
        _best_ap = _best_appr_pm.get(_proc, 'Baseline')
        _mode_order = sorted(_psub['SimMode'].unique(),
                             key=lambda m: _mwape.get((_proc, m), 999))

        # One sub-table per approach (Baseline + best)
        for _ap_key, _ap_tag in [('Baseline', 'Baseline'), (_best_ap, 'Best')]:
            _sub_ap = (_psub[_psub['Approach'] == _ap_key]
                       .groupby('SimMode')[_jpm_metrics].median())
            if _sub_ap.empty:
                continue
            _sub_ap = _sub_ap.reindex(_mode_order).round(4)
            _sub_ap.index = [_fmt_sim_mode(m) for m in _sub_ap.index]

            # Bold best (lowest) per column
            _sub_str = pd.DataFrame(index=_sub_ap.index, columns=_sub_ap.columns, dtype=object)
            for _c in _sub_ap.columns:
                _best = _sub_ap[_c].dropna().min()
                for _idx in _sub_ap.index:
                    _v = _sub_ap.loc[_idx, _c]
                    if pd.isna(_v):
                        _sub_str.loc[_idx, _c] = ''
                    else:
                        _fmt2 = f'{_v:.3f}'
                        _sub_str.loc[_idx, _c] = (r'\textbf{' + _fmt2 + r'}') if abs(_v - _best) < 1e-6 else _fmt2

            _proc_ltx = _proc.replace("_", r"\_")
            _ap_key_ltx = _ap_key.replace("_", r"\_")  # if you want LaTeX-escaped underscores

            _sub_str.columns = [
                r'\makecell{sMAE}' if c == 'sMAE' else
                r'\makecell{sRMSE}' if c == 'sRMSE' else
                r'\makecell{WAPE (\%)}' for c in _sub_str.columns
            ]

            _col_fmt = 'l|' + '|'.join(['c'] * len(_jpm_metrics))

            _ltx = _sub_str.to_latex(
                escape=False,
                column_format=_col_fmt,
                caption=(
                    f'Joint eval per simulation mode — {_proc_ltx}, '
                    f'curve approach: {_ap_tag} ({_ap_key_ltx}). '
                    r'Rows sorted by test duration WAPE (best mode first). '
                    r'Lower is better. \textbf{Bold} = best simulation mode.'
                ),
                label=f'tab:joint_pm_{EXPERIMENT}_{_proc}_{_ap_tag.lower()}',
                position='H',
            )

            print(_ltx)
            _pm_latex_parts.append(_ltx)

\begin{table}[H]
\caption{Joint eval per simulation mode — process\_1, curve approach: Baseline (Baseline). Rows sorted by test duration WAPE (best mode first). Lower is better. \textbf{Bold} = best simulation mode.}
\label{tab:joint_pm_505_process_1_baseline}
\begin{tabular}{l|c|c|c}
\toprule
 & \makecell{sMAE} & \makecell{sRMSE} & \makecell{WAPE (\%)} \\
\midrule
alpha / ml\_local & \textbf{1.000} & \textbf{1.010} & \textbf{95.413} \\
heuristic / ml\_local & \textbf{1.000} & \textbf{1.010} & \textbf{95.413} \\
inductive / ml\_local & \textbf{1.000} & \textbf{1.010} & \textbf{95.413} \\
alpha / ml\_global & \textbf{1.000} & \textbf{1.010} & \textbf{95.413} \\
heuristic / ml\_global & \textbf{1.000} & \textbf{1.010} & \textbf{95.413} \\
inductive / ml\_global & \textbf{1.000} & \textbf{1.010} & \textbf{95.413} \\
statistical & \textbf{1.000} & 1.012 & \textbf{95.413} \\
alpha / baseline & \textbf{1.000} & \textbf{1.010} & \textbf{95.413} \\
heuristic / baseline & \textbf{1.000} & \text

In [40]:
# ── COMBINED TABLE: process metrics + energy curve columns ────────────────────
# Same row structure as the main process-results table above.
# Three extra column groups added for Baseline and best curve approach:
#   sMAE | sRMSE | WAPE (energy)
# Source: curve_joint_duration_eval_per_mode.parquet (per-mode) if available,
#         falling back to curve_joint_duration_eval_results.parquet (best mode only).

_jpm_path  = run_dir / 'curve_joint_duration_eval_per_mode.parquet'
_jall_path = run_dir / 'curve_joint_duration_eval_results.parquet'

if _jpm_path.exists():
    _jsrc_comb = pd.read_parquet(_jpm_path)
    print(f'Using per-mode joint eval ({len(_jsrc_comb)} rows)')
elif _jall_path.exists():
    _jsrc_comb = pd.read_parquet(_jall_path)
    if 'BestMode' in _jsrc_comb.columns and 'SimMode' not in _jsrc_comb.columns:
        _jsrc_comb = _jsrc_comb.rename(columns={'BestMode': 'SimMode'})
    print(f'Per-mode file absent — using best-mode joint eval ({len(_jsrc_comb)} rows). '
          'Rows for non-best modes will show NaN in curve columns.')
else:
    _jsrc_comb = None
    print('No joint eval data found; run the pipeline first.')

if _jsrc_comb is not None:
    _comb_metrics = [m for m in ['sMAE', 'sRMSE', 'WAPE'] if m in _jsrc_comb.columns]

    # Pre-aggregate lookup: (Process, SimMode, Approach) → median curve metrics
    _jlkp = (
        _jsrc_comb
        .groupby(['Process', 'SimMode', 'Approach'])[_comb_metrics]
        .median()
    )

    # Best curve approach per process
    _stp2 = run_dir / 'summary_train_test.parquet'
    _best_ap_comb = {}
    if _stp2.exists():
        _stdf2 = pd.read_parquet(_stp2)
        for _pr2, _sg2 in _stdf2.groupby('Process'):
            _pa2 = _sg2.groupby('Approach')['WAPE_TEST'].median()
            if not _pa2.empty:
                _best_ap_comb[_pr2] = _pa2.idxmin()

    # Map (Process, ProcessModelLabel, DurationPredLabel) → raw SimMode string
    _pe_comb = pd.read_parquet(run_dir / 'process_eval_results.parquet')
    _ba_comb = {}
    for _pc in processes:
        _ba_comb[_pc] = pick_best_algo(_pe_comb[_pe_comb['process'] == _pc], split_prefix)

    def _label_to_simmode(proc, pmodel_lbl, dpred_lbl):
        ba = _ba_comb.get(proc, 'heuristic')
        if 'alpha' in pmodel_lbl:
            return 'petri_net_alpha'
        if 'ml_local'  in dpred_lbl: return f'petri_net_{ba}_ml_plus_per_act'
        if 'ml_global' in dpred_lbl: return f'petri_net_{ba}_ml_plus_global'
        return f'petri_net_{ba}'

    def _curve_vals(proc, sim_mode, approach):
        try:
            row = _jlkp.loc[(proc, sim_mode, approach)]
            return {m: row[m] for m in _comb_metrics}
        except KeyError:
            return {m: float('nan') for m in _comb_metrics}

    # ── Build extended table ──────────────────────────────────────────────────
    _ext_frames = []
    for proc in processes:
        sub_pe = _pe_comb[_pe_comb['process'] == proc]
        ba = _ba_comb[proc]
        best_ap = _best_ap_comb.get(proc, 'Baseline')
        pn = f'{ba} petri net'

        modes_to_show = ['petri_net_alpha', f'petri_net_{ba}',
                         f'petri_net_{ba}_ml_plus_global', f'petri_net_{ba}_ml_plus_per_act']
        sub_pe = sub_pe[sub_pe['mode'].isin(modes_to_show)].copy()
        sub_pe['mode_label'] = sub_pe['mode'].apply(lambda m: make_mode_label(m, ba))

        row_order = [f'{pn} / ml_local', f'{pn} / ml_global',
                     f'{pn} / baseline', 'alpha petri net / baseline']
        sub_pe = sub_pe.set_index('mode_label')[metric_cols]
        sub_pe = sub_pe.reindex([r for r in row_order if r in sub_pe.index])
        sub_pe.columns = [METRIC_LABELS.get(c, c) for c in sub_pe.columns]

        # Append curve metric columns for Baseline and best approach
        for _ck, _ctag in [('Baseline', 'Baseline'), (best_ap, 'Best')]:
            for _m in _comb_metrics:
                sub_pe[f'{_m} ({_ctag})'] = float('nan')

        for row_lbl in sub_pe.index:
            _pml, _dpl = split_mode(row_lbl)
            _sm = _label_to_simmode(proc, _pml, _dpl)
            for _ck, _ctag in [('Baseline', 'Baseline'), (best_ap, 'Best')]:
                _cv = _curve_vals(proc, _sm, _ck)
                for _m, _v in _cv.items():
                    sub_pe.loc[row_lbl, f'{_m} ({_ctag})'] = _v

        _pml_arr, _dpl_arr = zip(*[split_mode(r) for r in sub_pe.index])
        sub_pe.index = pd.MultiIndex.from_arrays(
            [[proc] * len(sub_pe), list(_pml_arr), list(_dpl_arr)],
            names=['Process', 'Process Model', 'Duration Pred.']
        )
        _ext_frames.append(sub_pe)

    ext_table = pd.concat(_ext_frames)
    display(ext_table)

    # ── LaTeX ─────────────────────────────────────────────────────────────────
    _higher_better_keys = {'F1 Score', 'Fitness', 'Precision'}

    def _is_hb_ext(col_label):
        return any(kw in col_label for kw in _higher_better_keys)

    _ext_str = pd.DataFrame(index=ext_table.index, columns=ext_table.columns, dtype=object)
    for proc in processes:
        _pmask = ext_table.index.get_level_values('Process') == proc
        _prows = ext_table[_pmask]
        for col in ext_table.columns:
            _vals = _prows[col].dropna()
            if _vals.empty:
                for idx in _prows.index: _ext_str.loc[idx, col] = ''
                continue
            _best_v = _vals.max() if _is_hb_ext(col) else _vals.min()
            for idx in _prows.index:
                _v2 = ext_table.loc[idx, col]
                if pd.isna(_v2):
                    _ext_str.loc[idx, col] = ''
                else:
                    _f2 = f'{_v2:.3f}'
                    _ext_str.loc[idx, col] = (r'\textbf{' + _f2 + r'}') if abs(_v2 - _best_v) < 1e-6 else _f2

    # Column format: process cols | baseline energy | best energy
    _n_proc   = len(metric_cols)
    _n_curve  = len(_comb_metrics)
    _col_fmt  = 'lll|' + '|'.join(['c'] * _n_proc) + '||' + '|'.join(['c'] * _n_curve) + '||' + '|'.join(['c'] * _n_curve)

    # Column header labels
    def _curve_col_label(m, tag):
        _nm = {'sMAE': r'sMAE', 'sRMSE': r'sRMSE', 'WAPE': r'WAPE\,(\%)'}
        return r'\makecell{' + _nm.get(m, m) + r'\\' + tag + r'}'

    _ext_str.columns = (
        [METRIC_LABELS.get(c, c) for c in metric_cols]
        + [_curve_col_label(m, 'Baseline') for m in _comb_metrics]
        + [_curve_col_label(m, 'Best')     for m in _comb_metrics]
    )

    _best_ap_label = ', '.join(sorted(set(_best_ap_comb.values())))
    _ext_latex = _ext_str.to_latex(
        multicolumn=True,
        multicolumn_format='c',
        multirow=True,
        escape=False,
        column_format=_col_fmt,
        caption=(
            f'Combined process and energy-curve results for experiment {EXPERIMENT} ({SPLIT} evaluation). '
            r'Left block: process model quality metrics (duration error, control-flow, conformance). '
            r'Middle block: energy curve quality using Baseline approach (mean barycenter, no DTW). '
            r'Right block: energy curve quality using the best curve approach per process '
            f'({_best_ap_label.replace("_", r"_")}). '
            r'Curve metrics are evaluated with simulated durations from each row\'s simulation mode '
            r'(shows how curve quality co-degrades with duration quality). '
            r'sMAE / sRMSE standardised by per-sensor std; WAPE is scale-free. '
            r'Lower is better for all metrics except Edge F1 Score, Fitness, Precision (higher = better). '
            r'\textbf{Bold} = best value per process per metric.'
        ),
        label=f'tab:combined_{EXPERIMENT}_{SPLIT}',
        position='H',
    )
    # Clean clines; insert midrules between processes
    _ext_latex = _re.sub(r'\s*\\cline\{[^}]+\}', '', _ext_latex)
    _ext_lines, _ext_new, _ext_cnt = _ext_latex.split('\n'), [], 0
    for _ln in _ext_lines:
        if _re.search(r'\\multirow.*\{process_', _ln):
            if _ext_cnt > 0: _ext_new.append(r'\midrule')
            _ext_cnt += 1
        _ext_new.append(_ln)
    _ext_latex = '\n'.join(_ext_new)
    print(_ext_latex.replace('_', r'\_'))

Using per-mode joint eval (117598 rows)


\makecell{Duration\\MAE (min)}  \
Process   Process Model       Duration Pred.                                   
process_1 heuristic petri net ml_local                              0.117859   
                              ml_global                             0.200576   
                              baseline                              0.985203   
          alpha petri net     baseline                              0.985203   
process_2 heuristic petri net ml_local                              4.507381   
                              ml_global                             3.013872   
                              baseline                              7.683950   
          alpha petri net     baseline                             27.125716   
process_3 heuristic petri net ml_local                              4.280978   
                              ml_global                             4.173959   
                              baseline                              3.772251   
          alpha petri net     baseline                             31.002561   
process_4 heuristic petri net ml_local                              4.091054   
                              ml_global                             3.595585   
                              baseline                              3.717815   
          alpha petri net     baseline                              8.199242   
process_5 heuristic petri net ml_local                              4.091054   
                              ml_global                             3.595585   
                              baseline                              3.717815   
          alpha petri net     baseline                              8.199242   

                                              \makecell{Duration\\WAPE}  \
Process   Process Model       Duration Pred.                              
process_1 heuristic petri net ml_local                         3.579618   
                              ml_global                        9.254257   
                              baseline                        33.081765   
          alpha petri net     baseline                        33.081765   
process_2 heuristic petri net ml_local                        24.700011   
                              ml_global                       22.189246   
                              baseline                        55.589791   
          alpha petri net     baseline                       103.685879   
process_3 heuristic petri net ml_local                        31.384028   
                              ml_global                       27.114529   
                              baseline                        33.856901   
          alpha petri net     baseline                        54.263526   
process_4 heuristic petri net ml_local                        39.705438   
                              ml_global                       34.896697   
                              baseline                        36.082991   
          alpha petri net     baseline                        44.121985   
process_5 heuristic petri net ml_local                        39.705438   
                              ml_global                       34.896697   
                              baseline                        36.082991   
          alpha petri net     baseline                        44.121985   

                                              \makecell{Duration\\RMSE (min)}  \
Process   Process Model       Duration Pred.                                    
process_1 heuristic petri net ml_local                               0.193549   
                              ml_global                              0.273868   
                              baseline                               1.602989   
          alpha petri net     baseline                               1.602989   
process_2 heuristic petri net ml_local                               8.831194   
                              ml_global                            

\begin{table}[H]
\caption{Combined process and energy-curve results for experiment 505 (test evaluation). Left block: process model quality metrics (duration error, control-flow, conformance). Middle block: energy curve quality using Baseline approach (mean barycenter, no DTW). Right block: energy curve quality using the best curve approach per process (DTW + Ext. Factors + Prev Act, DTW + Seq2Seq, DTW + pos). Curve metrics are evaluated with simulated durations from each row\'s simulation mode (shows how curve quality co-degrades with duration quality). sMAE / sRMSE standardised by per-sensor std; WAPE is scale-free. Lower is better for all metrics except Edge F1 Score, Fitness, Precision (higher = better). \textbf{Bold} = best value per process per metric.}
\label{tab:combined\_505\_test}
\begin{tabular}{lll|c|c|c|c|c|c||c|c|c||c|c|c}
\toprule
 &  &  & \makecell{Duration\\MAE (min)} & \makecell{Duration\\WAPE} & \makecell{Duration\\RMSE (min)} & \makecell{Edge\\F1 Score} & \makecell{